In [2]:

import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("cuDNN available:", tf.test.is_built_with_gpu_support())
print("GPU Devices:", tf.config.list_physical_devices('GPU'))

import torch
print("PyTorch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

TensorFlow version: 2.10.1
Built with CUDA: True
cuDNN available: True
GPU Devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
PyTorch version: 2.9.0+cpu
CUDA version: None
cuDNN version: None


In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import Sequence
from sklearn.metrics import classification_report

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10
DATA_ROOT = "datasets"

In [ ]:
class TxtDataset(Sequence):
    def __init__(self, txt_file, base_dir, batch_size=32, img_size=(224,224), shuffle=True):
        self.samples = [line.strip() for line in open(txt_file).readlines() if line.strip()]
        self.base_dir = base_dir
        self.batch_size = batch_size
        self.img_size = img_size
        self.shuffle = shuffle
        self.on_epoch_end()
    
    def __len__(self):
        # ensure the last smaller batch is not skipped
        return int(np.ceil(len(self.samples) / self.batch_size))

    def __getitem__(self, idx):
        batch = self.samples[idx * self.batch_size : (idx + 1) * self.batch_size]
        images, labels = [], []
        for line in batch:
            try:
                img_path, label = line.split()
                full_path = os.path.join(self.base_dir, img_path)
                img = cv2.imread(full_path)

                if img is None:
                    print(f"[Warning] Skipping unreadable image: {full_path}")
                    continue

                # convert and resize
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, self.img_size)
                img = img.astype(np.float32) / 255.0

                images.append(img)
                labels.append(int(label))
            except Exception as e:
                print(f"[Error in line: {line}] -> {e}")
                continue

        # if batch ended up empty, create dummy data to prevent crash
        if len(images) == 0:
            images = np.zeros((self.batch_size, *self.img_size, 3), dtype=np.float32)
            labels = np.zeros((self.batch_size,), dtype=np.float32)

        return np.array(images), np.array(labels)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.samples)

In [ ]:
train_gen = TxtDataset("D:\\codes\\project\\models\\datasets\\train.txt", DATA_ROOT, batch_size=BATCH_SIZE)
val_gen   = TxtDataset("D:\\codes\\project\\models\\datasets\\val.txt", DATA_ROOT, batch_size=BATCH_SIZE)
test_gen  = TxtDataset("D:\\codes\\project\\models\\datasets\\test.txt", DATA_ROOT, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))
x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(base_model.input, output)
for layer in base_model.layers:
    layer.trainable = False  # freeze base layers initially

model.compile(optimizer=Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
x, y = train_gen[0]
print(x.shape, y.shape)

In [ ]:
history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS)

In [ ]:
import cv2
from tensorflow.keras.models import load_model

model = load_model("fatigue_model.h5")

cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    img = cv2.resize(frame, (224,224)) / 255.0
    img = np.expand_dims(img, axis=0)
    pred = model.predict(img)[0][0]
    label = "Drowsy" if pred > 0.5 else "Active"
    cv2.putText(frame, label, (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)
    cv2.imshow("Fatigue Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()
